In [1]:
!pip install pandas nltk vaderSentiment matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Load Data


reviews_df = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\AMDARI\Montra Jupyter\reviews_with_topics.csv")
topic_df = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\AMDARI\Montra Jupyter\topic_summary.csv")




In [4]:
reviews_df.head(1)

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,clean_review,sentiment_encoded,word_count,char_count,stay_duration_calc,price_per_night,expected_sentiment,derived_sentiment,Topic,Topic_Probability
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,the overall price be fair for what we pay migh...,0,11,51,1,296.44,neutral,neutral,0,0.49


In [5]:
topic_df.head(13)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1661,-1_spa_receptionist_overall_pay,"['spa', 'receptionist', 'overall', 'pay', 'cle...",['the restaurant leave a lot to be desire seve...
1,0,9403,0_overall_pay_check_hotel,"['overall', 'pay', 'check', 'hotel', 'staff', ...",['overall our arrival experience be acceptable...
2,1,9089,1_time_hotel_stay_fine,"['time', 'hotel', 'stay', 'fine', 'glasgow', '...",['this hotel be a fine if unremarkable choice ...
3,2,3586,2_buffet_option_cafe_house cafe,"['buffet', 'option', 'cafe', 'house cafe', 'ho...",['arrival be unremarkable I would describe the...
4,3,1950,3_gym_fitness_fitness facility_facility,"['gym', 'fitness', 'fitness facility', 'facili...","['the fitness facility be standard', 'the fitn..."
5,4,1596,4_belfast_stay belfast_visit belfast_hotel,"['belfast', 'stay belfast', 'visit belfast', '...",['this hotel be a fine if unremarkable choice ...
6,5,1419,5_bathroom_bathroom area_area_clean,"['bathroom', 'bathroom area', 'area', 'clean',...",['sadly the pricing turn out to be not worth w...
7,6,1147,6_pool_overall pool_stay_overall,"['pool', 'overall pool', 'stay', 'overall', 'p...",['the rate we pay be also the pool be basic bu...
8,7,1081,7_fi_wi_wi fi_overall wi fi,"['fi', 'wi', 'wi fi', 'overall wi fi', 'overal...",['the neighborhood be fine on top of that the ...
9,8,984,8_base_ask base_base visit_ask base visit,"['base', 'ask base', 'base visit', 'ask base v...",['we could not have ask for a well base while ...


In [6]:
# merge tables

rev_top_df = pd.merge(reviews_df, topic_df, on='Topic', how='left')


In [7]:
rev_top_df.head(1)

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,stay_duration_calc,price_per_night,expected_sentiment,derived_sentiment,Topic,Topic_Probability,Count,Name,Representation,Representative_Docs
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,1,296.44,neutral,neutral,0,0.49,9403,0_overall_pay_check_hotel,"['overall', 'pay', 'check', 'hotel', 'staff', ...",['overall our arrival experience be acceptable...


In [8]:
rev_top_df['Topic'].value_counts().head()

Topic
 0    9403
 1    9089
 2    3586
 3    1950
-1    1661
Name: count, dtype: int64

In [9]:
import pandas as pd

# Mapping of topic IDs to meaningful aspect names
aspect_mapping = {
    "-1": "Spa & Reception",
    "0": "Overall Hotel Experience",
    "1": "Stay Experience",
    "2": "Restaurant & Buffet",
    "3": "Gym & Fitness",
    "4": "Belfast Location",
    "5": "Bathroom Cleanliness",
    "6": "Pool",
    "7": "Wi-Fi",
    "8": "General Visit",
    "9": "Carpet Cleanliness",
    "10": "Breakfast",
    "11": "Lounge"
}

# Extract topic number and map to aspect
rev_top_df["Aspect"] = (
    rev_top_df["Name"]
    .str.extract(r'^(-?\d+)')[0]
    .map(aspect_mapping)
)

print(rev_top_df[["Name", "Aspect"]].drop_duplicates())

                                          Name                    Aspect
0                    0_overall_pay_check_hotel  Overall Hotel Experience
2             6_pool_overall pool_stay_overall                      Pool
6    8_base_ask base_base visit_ask base visit             General Visit
7                       1_time_hotel_stay_fine           Stay Experience
8      3_gym_fitness_fitness facility_facility             Gym & Fitness
12  10_breakfast_overall breakfast_overall_pay                 Breakfast
17        11_lounge_overall lounge_overall_pay                    Lounge
20         5_bathroom_bathroom area_area_clean      Bathroom Cleanliness
31                 7_fi_wi_wi fi_overall wi fi                     Wi-Fi
33             2_buffet_option_cafe_house cafe       Restaurant & Buffet
36  4_belfast_stay belfast_visit belfast_hotel          Belfast Location
41             -1_spa_receptionist_overall_pay           Spa & Reception
84           9_carpet_clean_overall carpet_pay     

In [10]:
merge_map = {
    'Overall Hotel Experience': 'Overall Experience',
    'Stay Experience': 'Overall Experience',
    'General Visit': 'Overall Experience'
}

rev_top_df['Aspect'] = rev_top_df['Aspect'].replace(merge_map)

In [11]:
rev_top_df[["Name", "Aspect"]].drop_duplicates()

,Name,Aspect
0,0_overall_pay_check_hotel,Overall Experience
2,6_pool_overall pool_stay_overall,Pool
6,8_base_ask base_base visit_ask base visit,Overall Experience
7,1_time_hotel_stay_fine,Overall Experience
8,3_gym_fitness_fitness facility_facility,Gym & Fitness
12,10_breakfast_overall breakfast_overall_pay,Breakfast
17,11_lounge_overall lounge_overall_pay,Lounge
20,5_bathroom_bathroom area_area_clean,Bathroom Cleanliness
31,7_fi_wi_wi fi_overall wi fi,Wi-Fi
33,2_buffet_option_cafe_house cafe,Restaurant & Buffet


In [12]:
rev_top_df['Aspect'].unique()

array(['Overall Experience', 'Pool', 'Gym & Fitness', 'Breakfast',
       'Lounge', 'Bathroom Cleanliness', 'Wi-Fi', 'Restaurant & Buffet',
       'Belfast Location', 'Spa & Reception', 'Carpet Cleanliness'],
      dtype=object)

In [13]:
# Initialize sentimentanalyzer

analyzer = SentimentIntensityAnalyzer()

In [14]:
# Create sentiment function

def get_sentiment(text):
    
    score = analyzer.polarity_scores(str(text))
    compound = score['compound']
    
    if compound >= 0.05:
        sentiment = 'Positive'
    elif compound <= -0.05:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'
        
    return pd.Series([compound, sentiment])

In [15]:
# Analyze sentiment for each review

rev_top_df[['sentiment_score','sentiment']] = rev_top_df['clean_review'].apply(get_sentiment)

In [16]:
# check result

rev_top_df[['clean_review','Name','Aspect','sentiment_score','sentiment']].head(13)

,clean_review,Name,Aspect,sentiment_score,sentiment
0,the overall price be fair for what we pay migh...,0_overall_pay_check_hotel,Overall Experience,0.2263,Positive
1,the room be nothing out of the ordinary for wh...,0_overall_pay_check_hotel,Overall Experience,0.7269,Positive
2,unfortunately the pool be outdate it clearly n...,6_pool_overall pool_stay_overall,Pool,-0.2047,Negative
3,we be disappoint that the neighborhood be disa...,0_overall_pay_check_hotel,Overall Experience,-0.8807,Negative
4,the room service be standard hotel fare for wh...,0_overall_pay_check_hotel,Overall Experience,-0.1027,Negative
5,get around from the hotel be in a rather unsaf...,0_overall_pay_check_hotel,Overall Experience,-0.5399,Negative
6,we could not have ask for a well base while vi...,8_base_ask base_base visit_ask base visit,Overall Experience,0.8519,Positive
7,we have a wonderful time during our stay in no...,1_time_hotel_stay_fine,Overall Experience,0.7579,Positive
8,the in house cafe be acceptable for what we pa...,3_gym_fitness_fitness facility_facility,Gym & Fitness,-0.0387,Neutral
9,the manager be fine nothing special for what w...,3_gym_fitness_fitness facility_facility,Gym & Fitness,-0.2570,Negative


In [17]:
# Aggregate by topic Name

topic_sentiment = rev_top_df.groupby(['Aspect','sentiment']) \
                            .size() \
                            .reset_index(name='count')

print(topic_sentiment)

                  Aspect sentiment  count
0   Bathroom Cleanliness  Negative    454
1   Bathroom Cleanliness   Neutral     80
2   Bathroom Cleanliness  Positive    885
3       Belfast Location  Negative    313
4       Belfast Location   Neutral    100
5       Belfast Location  Positive   1183
6              Breakfast  Negative    233
7              Breakfast   Neutral     47
8              Breakfast  Positive    546
9     Carpet Cleanliness  Negative    197
10    Carpet Cleanliness   Neutral     47
11    Carpet Cleanliness  Positive    695
12         Gym & Fitness  Negative    469
13         Gym & Fitness   Neutral     99
14         Gym & Fitness  Positive   1382
15                Lounge  Negative    226
16                Lounge   Neutral     52
17                Lounge  Positive    539
18    Overall Experience  Negative   4490
19    Overall Experience   Neutral   1258
20    Overall Experience  Positive  13728
21                  Pool  Negative    329
22                  Pool   Neutral

In [26]:
rev_top_df = (
    rev_top_df.groupby(['Aspect', 'sentiment'])
      .size()
      .reset_index(name='count')
)

In [28]:
rev_top_df.columns

Index(['Aspect', 'sentiment', 'count'], dtype='object')

In [29]:
rev_top_df.head(1)

,Aspect,sentiment,count
0,Bathroom Cleanliness,Negative,454


In [30]:
summary = pd.pivot_table(
    rev_top_df,
    values="count",
    index="Aspect",
    columns="sentiment",
    aggfunc="sum",
    fill_value=0
)

summary["Total"] = summary.sum(axis=1)

for sentiment in ["Positive", "Negative", "Neutral"]:
    summary[f"{sentiment}_%"] = (
        summary[sentiment] / summary["Total"] * 100
    ).round(1)

summary = summary.sort_values("Negative_%", ascending=False)

summary

sentiment,Negative,Neutral,Positive,Total,Positive_%,Negative_%,Neutral_%
Aspect,,,,,,,
Bathroom Cleanliness,454,80,885,1419,62.4,32.0,5.6
Spa & Reception,484,97,1080,1661,65.0,29.1,5.8
Pool,329,66,752,1147,65.6,28.7,5.8
Breakfast,233,47,546,826,66.1,28.2,5.7
Wi-Fi,301,56,724,1081,67.0,27.8,5.2
Lounge,226,52,539,817,66.0,27.7,6.4
Restaurant & Buffet,921,217,2448,3586,68.3,25.7,6.1
Gym & Fitness,469,99,1382,1950,70.9,24.1,5.1
Overall Experience,4490,1258,13728,19476,70.5,23.1,6.5
